# Build: Masked Language Model

**Module 14** — BERT & Bidirectional Pre-Training

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Let's build a small masked language model. This won't be BERT-scale (you'd need 16 TPUs for that), but it captures every key idea: tokenization, masking, Transformer encoder, and predicting masked tokens.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MiniMLM(nn.Module):
    """A small Masked Language Model using Transformer encoder."""
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, d_ff=512, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)  # predict masked tokens
    
    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.encoder(x)
        x = self.ln(x)
        logits = self.head(x)  # (B, T, vocab_size)
        return logits

def mask_tokens(input_ids, vocab_size, mask_token_id, mask_prob=0.15):
    """Apply BERT-style masking: 80% [MASK], 10% random, 10% unchanged."""
    labels = input_ids.clone()
    probability_matrix = torch.full(input_ids.shape, mask_prob)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # only compute loss on masked tokens
    
    # 80% -> [MASK]
    replace_mask = torch.bernoulli(torch.full(input_ids.shape, 0.8)).bool() & masked_indices
    input_ids[replace_mask] = mask_token_id
    
    # 10% -> random token
    random_mask = torch.bernoulli(torch.full(input_ids.shape, 0.5)).bool() & masked_indices & ~replace_mask
    random_tokens = torch.randint(vocab_size, input_ids.shape)
    input_ids[random_mask] = random_tokens[random_mask]
    
    # 10% -> unchanged (no action needed)
    return input_ids, labels

# Example usage
vocab_size = 1000
model = MiniMLM(vocab_size)

### Step 2: Simulate a batch of tokenized text


In [ ]:
input_ids = torch.randint(4, vocab_size, (4, 32))  # batch=4, seq_len=32
masked_input, labels = mask_tokens(input_ids.clone(), vocab_size, mask_token_id=3)

logits = model(masked_input)
loss = F.cross_entropy(logits.view(-1, vocab_size), labels.view(-1), ignore_index=-100)
print(f"MLM loss: {loss.item():.4f}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
